In [88]:
from pathlib import Path

directory = Path("sites")     # the folder to scan
remove = "wide_"          # the substring to strip from filenames

for path in directory.iterdir():
    if path.is_file() and remove in path.name:
        new_name = path.name.replace(remove, "")
        path.rename(path.with_name(new_name))
        print(f"{path.name} -> {new_name}")

In [89]:
import pandas as pd

usgs_df = pd.read_csv("sites/USGS-06808500_all_data.csv")
iwqis_df = pd.read_csv("sites/WQS0115_all_data.csv")

print(usgs_df.columns)
print(iwqis_df.columns)

Index(['time', 'site_id', 'temp_water', 'nitrate_con', 'discharge', 'stage'], dtype='str')
Index(['site_uid', 'datetime', 'nitrate_con', 'r_exc', 'm_exc', 'temp_water',
       'ph', 'spec_cond', 'diss_oxy_con', 'diss_oxy_sat', 'chloro_v',
       'chloro_con', 'turbi_mean', 'turbi_var', 'turbi_med', 'turbi_bes',
       'turbi_min', 'turbi_max', 'dts_temp_water', 'phosphate_con', 'ph_v2',
       'spec_cond_v2', 'diss_oxy_con_v2', 'diss_oxy_sat_v2', 'turbi_mean_v2'],
      dtype='str')


In [90]:
import pandas as pd

usgs_to_iwqis_full = {
    "time": "datetime",
    "site_id": "site_uid",
    "temp_water": "temp_water",
    "nitrate_con": "nitrate_con",
    "diss_oxy_con": "diss_oxy_con",
    "diss_oxy_sat": "diss_oxy_sat",
    "ph": "ph",
    "spec_cond": "spec_cond",
    # USGS-only columns (discharge, stage) have no IWQIS name
}

# apply to one site file
usgs_df = pd.read_csv("sites/USGS-05465500_all_data.csv")
usgs_df = usgs_df.rename(columns=usgs_to_iwqis_full)


In [91]:
# keep only filtered keeper WQ sites or USGS sites
usgs_metadata = pd.read_csv("usgs_site_metadata.csv")
usgs_uids = usgs_metadata["monitoring_location_id"].unique()
site_clean = pd.read_csv("../IWQIS_archive/site_clean.csv")

In [92]:
coords = usgs_metadata["geometry"].str.extract(r"POINT \(([-\d.]+) ([-\d.]+)\)").astype(float)
usgs_metadata["longitude"] = coords[0]
usgs_metadata["latitude"]  = coords[1]
print(usgs_metadata.longitude.iloc[0])
print(usgs_metadata.latitude.iloc[0])

-91.3743179848803
40.3936553481252


In [ ]:
keeper_ids = pd.read_csv("iwqis_site_metadata.csv").uid.unique().tolist()
for uid in usgs_uids:
    id = str(uid).replace("USGS-", "")
    cond = (site_clean.uid == id).any()
    if cond:
        keeper_ids.append(uid)
        site_clean["uid"] = site_clean["uid"].replace(id, uid)

site_clean = site_clean[site_clean.uid.isin(keeper_ids)]

In [94]:
# convert geoemetry to latitude and longitude, delete
coords = usgs_metadata["geometry"].str.extract(r"POINT \(([-\d.]+) ([-\d.]+)\)").astype(float)
usgs_metadata["longitude"] = coords[0]
usgs_metadata["latitude"]  = coords[1]
print(usgs_metadata.longitude.iloc[0])
print(usgs_metadata.latitude.iloc[0])
usgs_metadata = usgs_metadata.drop(columns=["geometry"])

-91.3743179848803
40.3936553481252


In [95]:
# compare longitude and latitudes
for uid in usgs_uids:
    cond = (site_clean.uid == uid).any()
    if cond:
        lat1 = float(usgs_metadata.loc[usgs_metadata.monitoring_location_id == uid, "latitude"].iloc[0])
        lon1 = float(usgs_metadata.loc[usgs_metadata.monitoring_location_id == uid, "longitude"].iloc[0])
        lat2 = float(site_clean.loc[site_clean.uid == uid, "latitude"].iloc[0])
        lon2 = float(site_clean.loc[site_clean.uid == uid, "longitude"].iloc[0])
        
        latdiff = lat1 - lat2
        londiff = lon1 - lon2
        print(f"{latdiff:.8f} , {londiff:.8f}")

-0.00001365 , 0.00002697
-0.00001157 , 0.00000092
-0.00006667 , -0.00113333
0.00017778 , -0.00134444
-0.00003889 , 0.00001667
0.00000556 , -0.00002222
-0.00002222 , 0.00003333
-0.00003611 , 0.00001944
-0.00055556 , 0.00127778
0.00021667 , 0.00004444
0.00012778 , 0.00022222
0.00091667 , -0.00006667
-0.00026700 , -0.00235600
-0.00001389 , 0.00000833
0.00001597 , -0.00019854
-0.00007778 , -0.00032778
-0.00052778 , -0.00014444
0.00015556 , -0.00022778
-0.00013889 , -0.00018333
-0.00930556 , 0.00841667


In [96]:
# replace iwqis lat, lon with usgs
overlap = [uid for uid in usgs_uids if (site_clean.uid == uid).any()]
for uid in overlap:
    lat1 = float(usgs_metadata.loc[usgs_metadata.monitoring_location_id == uid, "latitude"].iloc[0])
    lon1 = float(usgs_metadata.loc[usgs_metadata.monitoring_location_id == uid, "longitude"].iloc[0])
    lat2 = float(site_clean.loc[site_clean.uid == uid, "latitude"].iloc[0])
    lon2 = float(site_clean.loc[site_clean.uid == uid, "longitude"].iloc[0])
    
    site_clean.loc[site_clean.uid == uid, "latitude"] = lat1
    site_clean.loc[site_clean.uid == uid, "longitude"] = lon1
    

In [97]:
print(usgs_metadata.columns)
print(site_clean.columns)

Index(['Unnamed: 0', 'time_series_id', 'unit_of_measure', 'parameter_name',
       'parameter_code', 'statistic_id', 'hydrologic_unit_code', 'state_name',
       'last_modified', 'begin', 'end', 'begin_utc', 'end_utc',
       'computation_period_identifier', 'computation_identifier', 'thresholds',
       'sublocation_identifier', 'primary', 'monitoring_location_id',
       'web_description', 'parameter_description', 'parent_time_series_id',
       'longitude', 'latitude'],
      dtype='str')
Index(['uid', 'nickname', 'latitude', 'longitude', 'river', 'road', 'town',
       'state', 'draining_area', 'icao', 'description', 'funding',
       'operator_uid', 'colloc_uid', 'link_uid', 'downstream_uid',
       'upstream_uid', 'private', 'status', 'status_message', 'deployed_at',
       'discontinued_at'],
      dtype='str')


In [102]:
# create remaining rows
uid_outside = set(usgs_uids).difference(set(overlap))
usgs_metadata = usgs_metadata.rename(columns={"state_name" : "state", "monitoring_location_id" : "uid"})
new_rows = usgs_metadata[usgs_metadata.uid.isin(uid_outside)][["state", "uid", "latitude", "longitude"]].drop_duplicates()

print(len(uid_outside))
new_df = pd.concat([site_clean, new_rows], ignore_index=True)
print(new_df.columns)
print(new_df.shape)
print(new_df[new_df.uid.isin(uid_outside)].shape[0])
print(new_df[new_df.uid.isin(uid_outside)])

6
Index(['uid', 'nickname', 'latitude', 'longitude', 'river', 'road', 'town',
       'state', 'draining_area', 'icao', 'description', 'funding',
       'operator_uid', 'colloc_uid', 'link_uid', 'downstream_uid',
       'upstream_uid', 'private', 'status', 'status_message', 'deployed_at',
       'discontinued_at'],
      dtype='str')
(88, 22)
6
                     uid nickname   latitude  longitude river road town state  \
82         USGS-05474500      NaN  40.393655 -91.374318   NaN  NaN  NaN  Iowa   
83         USGS-05480603      NaN  42.410889 -94.141306   NaN  NaN  NaN  Iowa   
84         USGS-05451210      NaN  42.315306 -93.152194   NaN  NaN  NaN  Iowa   
85         USGS-05464500      NaN  41.971945 -91.667124   NaN  NaN  NaN  Iowa   
86  USGS-415959091441301      NaN  41.999611 -91.736917   NaN  NaN  NaN  Iowa   
87         USGS-05480986      NaN  42.493250 -93.766806   NaN  NaN  NaN  Iowa   

    draining_area icao  ... operator_uid colloc_uid link_uid downstream_uid  \
82     

In [19]:
import pandas as pd

usgs = pd.read_csv("usgs-site-metadata.csv")
iwqis = pd.read_csv("../IWQIS-archive/site_clean.csv")

# rename uid -> site_uid to match the IWQIS data files
iwqis = iwqis.rename(columns={"uid": "site_uid"})

# bare join key on both sides (prefix stripped only for matching)
usgs["site_key"] = usgs["monitoring_location_id"].str.replace("USGS-", "", regex=False).str.strip()
iwqis["site_key"] = iwqis["site_uid"].astype(str).str.strip()

# collapse USGS metadata to one row per site; keep the PREFIXED id as usgs_id
usgs_site_level = (usgs
    .groupby("site_key")
    .agg(
        usgs_id=("monitoring_location_id", "first"),   # keeps "USGS-05474500"
        params=("parameter_code", lambda s: sorted(set(s.astype(str)))),
        n_params=("parameter_code", "nunique"),
    )
    .reset_index())

# left join on the bare key
merged = usgs_site_level.merge(
    iwqis, on="site_key", how="left",
    indicator=True, suffixes=("_usgs", "_iwqis"),
)

# drop the bare helper key; usgs_id (prefixed) is the anchor identifier
merged = merged.drop(columns="site_key")

print(merged.columns)
print(merged["_merge"].value_counts())
print("\nUSGS sites with no IWQIS match:")
print(merged.loc[merged["_merge"] == "left_only", "usgs_id"].tolist())

Index(['usgs_id', 'params', 'n_params', 'site_uid', 'nickname', 'latitude',
       'longitude', 'river', 'road', 'town', 'state', 'draining_area', 'icao',
       'description', 'funding', 'operator_uid', 'colloc_uid', 'link_uid',
       'downstream_uid', 'upstream_uid', 'private', 'status', 'status_message',
       'deployed_at', 'discontinued_at', '_merge'],
      dtype='str')
_merge
both          21
left_only      6
right_only     0
Name: count, dtype: int64

USGS sites with no IWQIS match:
['USGS-05451210', 'USGS-05464500', 'USGS-05474500', 'USGS-05480603', 'USGS-05480986', 'USGS-415959091441301']


In [107]:
def create_site_locations():
    # keep only filtered keeper WQ sites or USGS sites
    usgs_metadata = pd.read_csv("usgs_site_metadata.csv")
    site_clean = pd.read_csv("../IWQIS_archive/site_clean.csv")

    # modify the usgs_metadata column names
    usgs_metadata = usgs_metadata.rename(columns={"state_name": "state", "monitoring_location_id": "uid"})

    # convert geoemetry to latitude and longitude, delete
    coords = usgs_metadata["geometry"].str.extract(r"POINT \(([-\d.]+) ([-\d.]+)\)").astype(float)
    usgs_metadata["longitude"] = coords[0]
    usgs_metadata["latitude"] = coords[1]
    usgs_metadata = usgs_metadata.drop(columns=["geometry"])

    # get uids in and outside the site_clean metadata
    uid_overlap = [uid for uid in usgs_metadata.uid.unique() if (site_clean.uid == str(uid).replace("USGS-", "")).any()]
    uid_diff = set(usgs_metadata.uid.unique()).difference(set(uid_overlap))

    # replace the uids, latitudes and longitudes in site_clean \cap usgs_metadata
    for uid in uid_overlap:
        # replace uid first to make it less annoying
        site_clean.loc[site_clean.uid == str(uid).replace("USGS-", ""), "uid"] = uid
        
        lat1 = float(usgs_metadata.loc[usgs_metadata.uid == uid, "latitude"].iloc[0])
        lon1 = float(usgs_metadata.loc[usgs_metadata.uid == uid, "longitude"].iloc[0])
        lat2 = float(site_clean.loc[site_clean.uid == uid, "latitude"].iloc[0])
        lon2 = float(site_clean.loc[site_clean.uid == uid, "longitude"].iloc[0])

        site_clean.loc[site_clean.uid == uid, "latitude"] = lat1
        site_clean.loc[site_clean.uid == uid, "longitude"] = lon1

    keeper_ids = pd.read_csv("iwqis_site_metadata.csv").uid.unique().tolist()
    site_clean = site_clean[site_clean.uid.isin(keeper_ids + uid_overlap)]
    # get the final rows
    new_rows = usgs_metadata[usgs_metadata.uid.isin(uid_diff)][
        ["state", "uid", "latitude", "longitude"]
    ].drop_duplicates()

    # create the new locations file
    new_df = pd.concat([site_clean, new_rows], ignore_index=True)
    return new_df
    
df = create_site_locations()

print(df.shape)
print(df.columns)

(88, 22)
Index(['uid', 'nickname', 'latitude', 'longitude', 'river', 'road', 'town',
       'state', 'draining_area', 'icao', 'description', 'funding',
       'operator_uid', 'colloc_uid', 'link_uid', 'downstream_uid',
       'upstream_uid', 'private', 'status', 'status_message', 'deployed_at',
       'discontinued_at'],
      dtype='str')


In [108]:

def create_site_locations():
    # keep only filtered keeper WQ sites or USGS sites
    usgs_metadata = pd.read_csv("usgs_site_metadata.csv")
    site_clean = pd.read_csv("../IWQIS_archive/site_clean.csv")

    # modify the usgs_metadata column names
    usgs_metadata = usgs_metadata.rename(columns={"state_name": "state", "monitoring_location_id": "uid"})

    # convert geoemetry to latitude and longitude, delete
    coords = usgs_metadata["geometry"].str.extract(r"POINT \(([-\d.]+) ([-\d.]+)\)").astype(float)
    usgs_metadata["longitude"] = coords[0]
    usgs_metadata["latitude"] = coords[1]
    usgs_metadata = usgs_metadata.drop(columns=["geometry"])

    # get uids in and outside the site_clean metadata
    uid_overlap = [uid for uid in usgs_metadata.uid.unique() if (site_clean.uid == str(uid).replace("USGS-", "")).any()]
    uid_diff = set(usgs_metadata.uid.unique()).difference(set(uid_overlap))

    # replace the uids, latitudes and longitudes in site_clean \cap usgs_metadata
    for uid in uid_overlap:
        site_clean.loc[site_clean.uid == str(uid).replace("USGS-", ""), "uid"] = uid

        lat1 = float(usgs_metadata.loc[usgs_metadata.uid == uid, "latitude"].iloc[0])
        lon1 = float(usgs_metadata.loc[usgs_metadata.uid == uid, "longitude"].iloc[0])
        lat2 = float(site_clean.loc[site_clean.uid == uid, "latitude"].iloc[0])
        lon2 = float(site_clean.loc[site_clean.uid == uid, "longitude"].iloc[0])

        site_clean.loc[site_clean.uid == uid, "latitude"] = lat1
        site_clean.loc[site_clean.uid == uid, "longitude"] = lon1

    # remove the bad sites from site_clean
    keeper_ids = pd.read_csv("iwqis_site_metadata.csv").uid.unique().tolist()
    site_clean = site_clean[site_clean.uid.isin(keeper_ids + uid_overlap)]

    # get the final rows
    new_rows = usgs_metadata[usgs_metadata.uid.isin(uid_diff)][
        ["state", "uid", "latitude", "longitude"]
    ].drop_duplicates()

    # create the new locations file
    new_df = pd.concat([site_clean, new_rows], ignore_index=True)
    return new_df

new_df = create_site_locations()
print(new_df.shape)

(88, 22)
